In [ ]:
# expected: skip-floor
# Trivial setup; compute is well below the 10ms floor.
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
from collections import Counter
from dataclasses import dataclass
print("numpy", np.__version__, "pandas", pd.__version__, "scipy", scipy.__version__)

In [ ]:
# expected: cache
# Corner: DataFrame -> dataframe_numeric family; but columns are object
# strings (synthetic ticket subjects) so actual restore cost is much
# higher than the model's numeric-fit prediction.
_WORDS = ("account", "login", "reset", "password", "invoice", "refund",
          "subscription", "billing", "shipping", "delivery", "tracking",
          "support", "agent", "urgent", "priority", "escalate",
          "feature", "bug", "crash", "error", "timeout", "slow",
          "mobile", "desktop", "browser", "api", "integration", "webhook")

def _build_tickets(n_rows, seed=42):
    rng = np.random.default_rng(seed)
    lengths = rng.integers(40, 80, size=n_rows)
    words_arr = np.array(_WORDS)
    subjects = [" ".join(words_arr[rng.integers(0, len(_WORDS), size=L)])
                for L in lengths]
    return pd.DataFrame({
        "ticket_id": np.arange(n_rows, dtype=np.int64),
        "subject": subjects,
        "category": rng.choice(["billing", "tech", "shipping", "account"], n_rows),
        "priority": rng.choice(["low", "med", "high", "urgent"], n_rows),
    })

tickets = _build_tickets(30_000)
print(f"tickets: {tickets.shape}, mem: {tickets.memory_usage(deep=True).sum()/1e6:.0f} MB")

In [ ]:
# expected: cache
# Corner: dict -> dict_shallow family; but this is a doc-id -> token ->
# count nested map, so per-entry overhead is much higher than the
# shallow-dict fit assumes.
def _build_token_counts(ids, subjects):
    out = {}
    for tid, subj in zip(ids, subjects):
        out[int(tid)] = dict(Counter(subj.split()))
    return out

token_counts = _build_token_counts(tickets["ticket_id"].to_numpy(),
                                    tickets["subject"].to_numpy())
n_pairs = sum(len(v) for v in token_counts.values())
print(f"token_counts: {len(token_counts)} docs, {n_pairs} (doc,token) pairs")

In [ ]:
# expected: cache
# Corner: Categorical isn't in _TYPE_TO_FAMILY, but it lives inside a
# DataFrame so the DF routing still applies. Tests whether Categorical
# storage cost is captured by the numeric-DF fit.
def _encode(df):
    out = df.copy()
    out["category"] = out["category"].astype("category")
    out["priority"] = out["priority"].astype("category")
    out["subject_len"] = out["subject"].str.len().astype(np.int32)
    return out

encoded = _encode(tickets)
summary_by_cat = encoded.groupby(["category", "priority"], observed=True).agg(
    n=("ticket_id", "count"),
    avg_len=("subject_len", "mean"),
).reset_index()
print(f"encoded: {encoded.shape}; summary_by_cat: {summary_by_cat.shape}")

In [ ]:
# expected: cache
# Corner: sparse family has R^2=0.516 for disk-deserialize -- the worst
# fit in the model. Densify deliberately to stress the linear-in-bytes
# assumption.
def _build_tfidf(token_counts):
    vocab = sorted({w for inner in token_counts.values() for w in inner})
    vocab_idx = {w: i for i, w in enumerate(vocab)}
    rows, cols, vals = [], [], []
    for doc_idx, inner in enumerate(token_counts.values()):
        for w, c in inner.items():
            rows.append(doc_idx)
            cols.append(vocab_idx[w])
            vals.append(c)
    mat = sp.csr_matrix(
        (np.asarray(vals, dtype=np.float32),
         (np.asarray(rows, dtype=np.int32),
          np.asarray(cols, dtype=np.int32))),
        shape=(len(token_counts), len(vocab)),
    )
    return mat, vocab

tfidf_like, vocab = _build_tfidf(token_counts)
density = tfidf_like.nnz / (tfidf_like.shape[0] * tfidf_like.shape[1])
print(f"tfidf_like: shape={tfidf_like.shape}, nnz={tfidf_like.nnz}, density={density:.4f}")

In [ ]:
# expected: cache
# Corner: FeatureBundle is a custom class -> _GENERIC fallback. Tests
# whether the "slowest observed" conservatism actually covers a mixed
# payload of DataFrame + sparse + dict. The .copy() calls make the
# bundle own deep payloads so cash sees a substantive object and the
# construction crosses the 10ms execution floor.
@dataclass
class FeatureBundle:
    df: pd.DataFrame
    matrix: sp.csr_matrix
    counts: dict

def _make_bundle(df, matrix, counts):
    return FeatureBundle(
        df=df.copy(),
        matrix=matrix.copy(),
        counts={k: dict(v) for k, v in counts.items()},
    )

bundle = _make_bundle(encoded, tfidf_like, token_counts)
print(f"bundle: type={type(bundle).__name__}, df={bundle.df.shape}, "
      f"matrix_nnz={bundle.matrix.nnz}, counts={len(bundle.counts)}")

In [ ]:
# expected: cache
# Sanity: classic "cache me" case -- seconds of compute, kilobytes of
# output. Policy MUST cache this.
def _do_eigh(dim, seed=7):
    rng = np.random.default_rng(seed)
    m = rng.standard_normal((dim, dim)).astype(np.float64)
    m = (m + m.T) / 2
    return np.linalg.eigh(m)

eigvals, eigvecs = _do_eigh(1000)
top_eigvals = eigvals[-10:]
print(f"eigvals: {eigvals.shape}, top10={top_eigvals}")

In [ ]:
# expected: skip-cost
# Corner: cheap compute (~5ms) producing 40MB of float64. Restore cost
# >> compute cost. Policy MUST refuse to cache via the cost-model ratio.
huge_array = np.ones((5_000_000,), dtype=np.float64)
print(f"huge_array: shape={huge_array.shape}, bytes={huge_array.nbytes/1e6:.0f} MB")

In [ ]:
# expected: skip-floor
# Corner: trivial filter; below the 10ms floor. Should be skipped by
# floor, NOT by cost model. (The four cost_model_* metric fields should
# be absent on this cell.)
preview = tickets.head(20)
print(f"preview: {preview.shape}")

In [ ]:
# expected: cache
# Corner: self-assignment pattern -- df = df.sort_values(); df = df.assign().
# Tests cost-model interaction with the self-assign skip-check path.
def _augment(df):
    out = df.sort_values("ticket_id").reset_index(drop=True)
    out = out.assign(
        subj_word_count=out["subject"].str.split().str.len().astype(np.int32),
    )
    return out

encoded = _augment(encoded)
print(f"encoded after self-assign chain: {encoded.shape}")

In [ ]:
# expected: cache
# Tests cache-hit propagation when upstream policy decisions vary
# (bundle from cell 6 is _GENERIC, tfidf_like from cell 5 is sparse).
def _summarize(encoded, tfidf_like):
    doc_norms = np.asarray(np.sqrt(tfidf_like.multiply(tfidf_like).sum(axis=1))).ravel()
    combined = encoded.assign(doc_norm=doc_norms[:len(encoded)])
    return combined.groupby("category", observed=True).agg(
        n=("ticket_id", "count"),
        avg_norm=("doc_norm", "mean"),
        avg_words=("subj_word_count", "mean"),
    ).reset_index()

final_summary = _summarize(encoded, tfidf_like)
print("final_summary:")
print(final_summary)

In [ ]:
# expected: skip-floor
assert final_summary.shape[0] > 0
print(f"OK: {final_summary.shape[0]} categories")